# N104 · 序列匹配、编辑距离与子序列计数

**目标：** 按两个前缀定义状态并区分最优值和方案数。

**先修：** N098, N022。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** LCS；编辑增删改；不同子序列；空串边界；路径恢复。

## 从问题到算法

二维前缀状态dp[i][j]描述两个序列的前i、j个元素。LCS选择是否对齐最后一项；编辑距离枚举最后一次操作；子序列计数则对“用或不用当前字符”求和。它们共享网格形状，却有不同代数：max、min、sum不能混用。

## 最小实现

**本章接口：** `lcs(a, b)`、`edit_distance(a, b)`、`num_distinct(s, t)`

In [1]:
def lcs(a,b):
    dp=[0]*(len(b)+1)
    for x in a:
        old=dp; dp=[0]*(len(b)+1)
        for j,y in enumerate(b,1): dp[j]=old[j-1]+1 if x==y else max(old[j],dp[j-1])
    return dp[-1]

def edit_distance(a,b):
    dp=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        old=dp; dp=[i]+[0]*len(b)
        for j,y in enumerate(b,1): dp[j]=old[j-1] if x==y else 1+min(old[j],dp[j-1],old[j-1])
    return dp[-1]

def num_distinct(s,t):
    dp=[1]+[0]*len(t)
    for x in s:
        for j in range(len(t),0,-1):
            if x==t[j-1]: dp[j]+=dp[j-1]
    return dp[-1]

## 正确性、前提与复杂度

前缀边界明确处理空序列。LCS相同末字符可以在某个最优解中对齐，不同末字符至少丢弃一端；编辑距离按最后操作分解；计数的逆序更新保证一个来源字符至多匹配目标中的一个位置。

**代价：** 各算法O(mn)算术操作、O(n)辅助空间。编辑距离使用单位插入/删除/替换成本；计数保留任意精度整数，输出位数影响实际代价。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a,b='abcde','ace'; dp=[[0]*(len(b)+1) for _ in range(len(a)+1)]
for i,x in enumerate(a,1):
    for j,y in enumerate(b,1): dp[i][j]=dp[i-1][j-1]+1 if x==y else max(dp[i-1][j],dp[i][j-1])
show_table(['a的前缀','b前缀的LCS长度'],[(a[:i],row) for i,row in enumerate(dp)])

a的前缀,b前缀的LCS长度
,"[0, 0, 0, 0]"
a,"[0, 1, 1, 1]"
ab,"[0, 1, 1, 1]"
abc,"[0, 1, 2, 2]"
abcd,"[0, 1, 2, 2]"
abcde,"[0, 1, 2, 3]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert lcs('abcde','ace')==3 and edit_distance('horse','ros')==3
assert num_distinct('rabbbit','rabbit')==3 and num_distinct('abc','')==1
from itertools import product,combinations
from functools import cache
words=[''.join(p) for n in range(5) for p in product('ab',repeat=n)]
def subs(s): return {''.join(s[i] for i in ids) for k in range(len(s)+1) for ids in combinations(range(len(s)),k)}
@cache
def edit_ref(a,b):
    if not a or not b: return len(a)+len(b)
    return min(edit_ref(a[1:],b)+1,edit_ref(a,b[1:])+1,edit_ref(a[1:],b[1:])+(a[0]!=b[0]))
for a in words:
    for b in words:
        assert lcs(a,b)==max(map(len,subs(a)&subs(b)))
        assert edit_distance(a,b)==edit_ref(a,b)
        assert num_distinct(a,b)==sum(''.join(a[i] for i in ids)==b for ids in combinations(range(len(a)),len(b)))
print('N104: 所有本章断言通过')

N104: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 恢复一条LCS。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释不同子序列对相同字符仍可能有多种索引选择。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1143. Longest Common Subsequence（canonical）
- 72. Edit Distance（canonical）
- 115. Distinct Subsequences（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。